In [17]:
%pip -q install msal requests

import msal
import requests

print("Librerías listas para conectar con Microsoft.")

Librerías listas para conectar con Microsoft.


In [48]:
CLIENT_ID = "893a0b41-a2d2-4254-b815-0b3768e9971d"
TENANT_ID = "249be59a-cd09-4c9d-a466-ae4e04421ad1"

SCOPES = ["User.Read", "Sites.Read.All"]
#SCOPES = ["User.Read", "Files.Read"]

token_acceso = None

app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}"
)

flujo = app.initiate_device_flow(scopes=SCOPES)

if "user_code" not in flujo:
    raise RuntimeError(
        flujo.get("error_description", "No se pudo generar el código.")
    )

print(flujo["message"], flush=True)

resultado = app.acquire_token_by_device_flow(flujo)

if "access_token" in resultado:
    token_acceso = resultado["access_token"]
    print("Inicio de sesión correcto.")
else:
    print("Error:", resultado.get("error"))
    print(resultado.get("error_description"))

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code FHC9NLQGT to authenticate.
Inicio de sesión correcto.


In [49]:
url_biblioteca = (
    "https://graph.microsoft.com/v1.0/sites/layaquesita302.sharepoint.com:/sites/PruebasPdfPython:/drive"
)

headers = {"Authorization": f"Bearer {token_acceso}"}
drive_id = None

respuesta = requests.get(
    url_biblioteca,
    headers=headers,
    params={"$select": "id,name,webUrl"},
    timeout=30
)

print("Estado HTTP:", respuesta.status_code)

if respuesta.status_code == 200:
    biblioteca = respuesta.json()
    drive_id = biblioteca["id"]

    print("Biblioteca:", biblioteca.get("name"))
    print("Dirección:", biblioteca.get("webUrl"))
    print("Identificador guardado en drive_id.")
else:
    print(respuesta.text)

Estado HTTP: 200
Biblioteca: Documentos
Dirección: https://layaquesita302.sharepoint.com/sites/PruebasPdfPython/Documentos%20compartidos
Identificador guardado en drive_id.


In [50]:
url_archivos = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root/children"

respuesta_archivos = requests.get(
    url_archivos,
    headers=headers,
    timeout=30
)

print("Estado HTTP:", respuesta_archivos.status_code)

if respuesta_archivos.status_code == 200:
    archivos = respuesta_archivos.json()["value"]

    for archivo in archivos:
        tipo = "Carpeta" if "folder" in archivo else "Archivo"
        print(f"{tipo}: {archivo['name']}")
else:
    print(respuesta_archivos.text)

Estado HTTP: 200
Archivo: 02_orden_mxn.pdf
Archivo: 03_orden_mxn.pdf
Archivo: 04_orden_mxn.pdf


In [51]:
from pathlib import Path

nombre_pdf = "02_orden_mxn.pdf"

# Buscar el PDF dentro del listado obtenido
pdf_elegido = next(
    archivo for archivo in archivos
    if archivo["name"] == nombre_pdf
)

# Solicitar el contenido del archivo
url_descarga = (
    f"https://graph.microsoft.com/v1.0/drives/{drive_id}"
    f"/items/{pdf_elegido['id']}/content"
)

respuesta_pdf = requests.get(
    url_descarga,
    headers=headers,
    timeout=60
)

print("Estado HTTP:", respuesta_pdf.status_code)

if respuesta_pdf.status_code == 200:
    carpeta_descargas = Path("pdfs_descargados")
    carpeta_descargas.mkdir(exist_ok=True)

    ruta_pdf = carpeta_descargas / nombre_pdf
    ruta_pdf.write_bytes(respuesta_pdf.content)

    print("PDF guardado en:", ruta_pdf.resolve())
    print("Tamaño en bytes:", ruta_pdf.stat().st_size)
else:
    print(respuesta_pdf.text)

Estado HTTP: 200
PDF guardado en: /content/pdfs_descargados/02_orden_mxn.pdf
Tamaño en bytes: 2324


In [21]:
%pip -q install pypdf

from pypdf import PdfReader

# Abrir el PDF que acabamos de descargar
lector = PdfReader(ruta_pdf)

print("Cantidad de páginas:", len(lector.pages))

# Extraer y juntar el texto de todas las páginas
texto_pdf = ""

for pagina in lector.pages:
    texto_pagina = pagina.extract_text() or ""
    texto_pdf += texto_pagina + "\n"

print("\nTexto extraído:\n")
print(texto_pdf)

Cantidad de páginas: 1

Texto extraído:

ORDEN DE COMPRA
PYTHON-PDF / DATOS FICTICIOS / NO ES UN DOCUMENTO COMERCIAL
Laboratorio de extraccion | Pagina 1 | Sin validez comercial
Orden de compra: OC-2002
Proveedor: Proveedor Ficticio Centro
Fecha: 2026-11-02
Moneda: MXN
Formato numerico: US
Subtotal: 850.75
Impuesto: 136.12
Total: 986.87
PN
Descripcion
Cantidad
Precio unitario
Importe
000127
Tornillo de prueba
2
250.25
500.50
000128
Arandela de prueba
1
350.25
350.25




In [22]:
import pandas as pd

datos_orden = {}

# Recorrer el texto línea por línea
for linea in texto_pdf.splitlines():

    # Identificar líneas con formato Campo: valor
    if ":" in linea:
        campo, valor = linea.split(":", 1)

        # Guardar el campo y su valor, quitando espacios sobrantes
        datos_orden[campo.strip()] = valor.strip()

# Convertir el diccionario en una tabla de una fila
df_orden = pd.DataFrame([datos_orden])

display(df_orden)

,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2002,Proveedor Ficticio Centro,2026-11-02,MXN,US,850.75,136.12,986.87


In [23]:
# Convertir los importes de texto a números
for columna in ["Subtotal", "Impuesto", "Total"]:
    df_orden[columna] = pd.to_numeric(
        df_orden[columna],
        errors="raise"
    )

# Convertir la fecha indicando el formato año-mes-día
df_orden["Fecha"] = pd.to_datetime(
    df_orden["Fecha"],
    format="%Y-%m-%d",
    errors="raise"
)

display(df_orden)

print("\nTipos de datos:")
print(df_orden.dtypes)

,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2002,Proveedor Ficticio Centro,2026-11-02,MXN,US,850.75,136.12,986.87



Tipos de datos:
Orden de compra             object
Proveedor                   object
Fecha               datetime64[ns]
Moneda                      object
Formato numerico            object
Subtotal                   float64
Impuesto                   float64
Total                      float64
dtype: object


In [24]:
# Separar el texto en líneas y quitar las vacías
lineas = [
    linea.strip()
    for linea in texto_pdf.splitlines()
    if linea.strip()
]

# Los productos comienzan después del encabezado "Importe"
inicio = lineas.index("Importe") + 1
lineas_productos = lineas[inicio:]

if len(lineas_productos) % 5 != 0:
    raise ValueError("Las líneas de productos no forman grupos de cinco.")

# Crear una fila por producto
filas = []

for i in range(0, len(lineas_productos), 5):
    pn, descripcion, cantidad, precio, importe = lineas_productos[i:i + 5]

    filas.append({
        "Orden de compra": df_orden.loc[0, "Orden de compra"],
        "PN": pn,
        "Descripcion": descripcion,
        "Cantidad": int(cantidad),
        "Precio unitario": float(precio),
        "Importe": float(importe)
    })

df_partidas = pd.DataFrame(filas)

display(df_partidas)

,Orden de compra,PN,Descripcion,Cantidad,Precio unitario,Importe
0,OC-2002,000127,Tornillo de prueba,2,250.25,500.50
1,OC-2002,000128,Arandela de prueba,1,350.25,350.25


In [25]:
# Revisar cantidad × precio unitario para cada producto
revision_partidas = df_partidas.copy()

revision_partidas["Importe calculado"] = (
    revision_partidas["Cantidad"]
    * revision_partidas["Precio unitario"]
).round(2)

revision_partidas["Coincide"] = (
    revision_partidas["Importe calculado"]
    == revision_partidas["Importe"].round(2)
)

display(
    revision_partidas[
        ["PN", "Importe", "Importe calculado", "Coincide"]
    ]
)

# Comprobar que las partidas suman el subtotal del PDF
subtotal_calculado = round(df_partidas["Importe"].sum(), 2)
subtotal_pdf = round(df_orden.loc[0, "Subtotal"], 2)

# Comprobar subtotal + impuesto = total
total_calculado = round(
    subtotal_pdf + df_orden.loc[0, "Impuesto"], 2
)
total_pdf = round(df_orden.loc[0, "Total"], 2)

print("Subtotal de las partidas:", subtotal_calculado)
print("Subtotal coincide:", subtotal_calculado == subtotal_pdf)

print("\nTotal calculado:", total_calculado)
print("Total coincide:", total_calculado == total_pdf)

,PN,Importe,Importe calculado,Coincide
0,000127,500.50,500.50,True
1,000128,350.25,350.25,True


Subtotal de las partidas: 850.75
Subtotal coincide: True

Total calculado: 986.87
Total coincide: True


In [26]:
# aqui ya se guarda en excel

%pip -q install openpyxl

from pathlib import Path
from google.colab import files

# Crear la carpeta de resultados
carpeta_salida = Path("resultados")
carpeta_salida.mkdir(exist_ok=True)

ruta_excel = carpeta_salida / "resultado_OC-2002.xlsx"

# Guardar ambas tablas en un mismo Excel
with pd.ExcelWriter(ruta_excel, engine="openpyxl") as escritor:
    df_orden.to_excel(
        escritor, sheet_name="Orden", index=False
    )

    df_partidas.to_excel(
        escritor, sheet_name="Partidas", index=False
    )

print("Excel creado:", ruta_excel.resolve())

# Descargarlo a tu computadora
files.download(str(ruta_excel))

Excel creado: /content/resultados/resultado_OC-2002.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
## aqui es un separador. empezo otra etapa por asi decirlo

In [28]:
def extraer_encabezado(texto):
    datos = {}

    for linea in texto.splitlines():
        if ":" in linea:
            campo, valor = linea.split(":", 1)
            datos[campo.strip()] = valor.strip()

    tabla = pd.DataFrame([datos])

    tabla["Fecha"] = pd.to_datetime(
        tabla["Fecha"],
        format="%Y-%m-%d",
        errors="raise"
    )

    for columna in ["Subtotal", "Impuesto", "Total"]:
        tabla[columna] = pd.to_numeric(
            tabla[columna],
            errors="raise"
        )

    return tabla


# Probar la función con el texto que ya tenemos
orden_prueba = extraer_encabezado(texto_pdf)

display(orden_prueba)

,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2002,Proveedor Ficticio Centro,2026-11-02,MXN,US,850.75,136.12,986.87


In [29]:
def extraer_partidas(texto, numero_orden):
    lineas = [
        linea.strip()
        for linea in texto.splitlines()
        if linea.strip()
    ]

    inicio = lineas.index("Importe") + 1
    lineas_productos = lineas[inicio:]

    if not lineas_productos or len(lineas_productos) % 5 != 0:
        raise ValueError(
            "Se esperaban productos en grupos de cinco líneas."
        )

    filas = []

    for i in range(0, len(lineas_productos), 5):
        pn, descripcion, cantidad, precio, importe = (
            lineas_productos[i:i + 5]
        )

        filas.append({
            "Orden de compra": numero_orden,
            "PN": pn,
            "Descripcion": descripcion,
            "Cantidad": int(cantidad),
            "Precio unitario": float(precio),
            "Importe": float(importe)
        })

    return pd.DataFrame(filas)


# Probar la función con nuestra orden
partidas_prueba = extraer_partidas(
    texto_pdf,
    orden_prueba.loc[0, "Orden de compra"]
)

display(partidas_prueba)

,Orden de compra,PN,Descripcion,Cantidad,Precio unitario,Importe
0,OC-2002,000127,Tornillo de prueba,2,250.25,500.50
1,OC-2002,000128,Arandela de prueba,1,350.25,350.25


In [30]:
from pypdf import PdfReader

def procesar_pdf(ruta_archivo):
    # Leer el texto del archivo
    lector = PdfReader(ruta_archivo)
    texto = ""

    for pagina in lector.pages:
        texto += (pagina.extract_text() or "") + "\n"

    if not texto.strip():
        raise ValueError("El PDF no devolvió texto extraíble.")

    # Extraer el encabezado
    encabezado = extraer_encabezado(texto)

    # Usar su número de orden para extraer las partidas
    numero_orden = encabezado.loc[0, "Orden de compra"]
    partidas = extraer_partidas(texto, numero_orden)

    return encabezado, partidas


# Procesar nuestro PDF con una sola llamada
orden_procesada, partidas_procesadas = procesar_pdf(ruta_pdf)

print("DATOS GENERALES")
display(orden_procesada)

print("PARTIDAS")
display(partidas_procesadas)

DATOS GENERALES


,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2002,Proveedor Ficticio Centro,2026-11-02,MXN,US,850.75,136.12,986.87


PARTIDAS


,Orden de compra,PN,Descripcion,Cantidad,Precio unitario,Importe
0,OC-2002,000127,Tornillo de prueba,2,250.25,500.50
1,OC-2002,000128,Arandela de prueba,1,350.25,350.25


In [52]:
url_archivos = (
    f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root/children"
)

respuesta_archivos = requests.get(
    url_archivos,
    headers=headers,
    timeout=30
)

print("Estado HTTP:", respuesta_archivos.status_code)

pdfs_sharepoint = []

if respuesta_archivos.status_code == 200:
    archivos = respuesta_archivos.json()["value"]

    pdfs_sharepoint = [
        archivo for archivo in archivos
        if "file" in archivo
        and archivo["name"].lower().endswith(".pdf")
    ]

    for archivo in pdfs_sharepoint:
        print("PDF encontrado:", archivo["name"])

    print("\nCantidad de PDFs:", len(pdfs_sharepoint))
else:
    print(respuesta_archivos.text)

Estado HTTP: 200
PDF encontrado: 02_orden_mxn.pdf
PDF encontrado: 03_orden_mxn.pdf
PDF encontrado: 04_orden_mxn.pdf

Cantidad de PDFs: 3


In [32]:
from pathlib import Path

carpeta_descargas = Path("pdfs_descargados")
carpeta_descargas.mkdir(exist_ok=True)

# Empezar con listas vacías en cada ejecución
encabezados_lote = []
partidas_lote = []

for archivo in pdfs_sharepoint:
    print("Procesando:", archivo["name"])

    # Descargar el PDF
    url_descarga = (
        f"https://graph.microsoft.com/v1.0/drives/{drive_id}"
        f"/items/{archivo['id']}/content"
    )

    descarga = requests.get(
        url_descarga,
        headers=headers,
        timeout=60
    )
    descarga.raise_for_status()

    ruta_local = carpeta_descargas / archivo["name"]
    ruta_local.write_bytes(descarga.content)

    # Obtener sus dos tablas usando nuestra función
    encabezado, partidas = procesar_pdf(ruta_local)

    encabezados_lote.append(encabezado)
    partidas_lote.append(partidas)

# Unir los resultados de todos los PDFs
df_ordenes = pd.concat(encabezados_lote, ignore_index=True)
df_partidas_consolidadas = pd.concat(partidas_lote, ignore_index=True)

print("\nÓrdenes procesadas:", len(df_ordenes))
display(df_ordenes)

print("\nPartidas procesadas:", len(df_partidas_consolidadas))
display(df_partidas_consolidadas)

Procesando: 02_orden_mxn.pdf
Procesando: 03_orden_mxn.pdf

Órdenes procesadas: 2


,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2002,Proveedor Ficticio Centro,2026-11-02,MXN,US,850.75,136.12,986.87
1,OC-2003,Proveedor Ficticio Norte,2026-11-03,MXN,US,976.00,156.16,1132.16



Partidas procesadas: 5


,Orden de compra,PN,Descripcion,Cantidad,Precio unitario,Importe
0,OC-2002,000127,Tornillo de prueba,2,250.25,500.50
1,OC-2002,000128,Arandela de prueba,1,350.25,350.25
2,OC-2003,000231,Tuerca de prueba,3,110.50,331.50
3,OC-2003,000232,Perno de prueba,2,245.75,491.50
4,OC-2003,000233,Separador de prueba,4,38.25,153.00


In [34]:
revisiones = []

for _, orden in df_ordenes.iterrows():
    numero_orden = orden["Orden de compra"]

    # Seleccionar las partidas de esta orden
    partidas_orden = df_partidas_consolidadas.loc[
        df_partidas_consolidadas["Orden de compra"] == numero_orden
    ]

    # Comprobar cantidad × precio unitario
    importes_calculados = (
        partidas_orden["Cantidad"]
        * partidas_orden["Precio unitario"]
    ).round(2)

    importes_correctos = (
        not partidas_orden.empty
        and (
            importes_calculados == partidas_orden["Importe"].round(2)
        ).all()
    )

    # Calcular subtotal y total
    subtotal_calculado = round(partidas_orden["Importe"].sum(), 2)
    total_calculado = round(orden["Subtotal"] + orden["Impuesto"], 2)

    revisiones.append({
        "Orden de compra": numero_orden,
        "Partidas": len(partidas_orden),
        "Importes correctos": bool(importes_correctos),
        "Subtotal correcto": subtotal_calculado == round(orden["Subtotal"], 2),
        "Total correcto": total_calculado == round(orden["Total"], 2)
    })

df_validacion_lote = pd.DataFrame(revisiones)

display(df_validacion_lote)

,Orden de compra,Partidas,Importes correctos,Subtotal correcto,Total correcto
0,OC-2002,2,True,True,True
1,OC-2003,3,True,True,True


In [35]:
from pathlib import Path
from google.colab import files

carpeta_salida = Path("resultados")
carpeta_salida.mkdir(exist_ok=True)

ruta_excel_lote = carpeta_salida / "ordenes_consolidadas.xlsx"

with pd.ExcelWriter(ruta_excel_lote, engine="openpyxl") as escritor:

    df_ordenes.to_excel(
        escritor,
        sheet_name="Ordenes",
        index=False
    )

    df_partidas_consolidadas.to_excel(
        escritor,
        sheet_name="Partidas",
        index=False
    )

    df_validacion_lote.to_excel(
        escritor,
        sheet_name="Validaciones",
        index=False
    )

print("Excel creado:", ruta_excel_lote.resolve())

files.download(str(ruta_excel_lote))

Excel creado: /content/resultados/ordenes_consolidadas.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
inventario = []

for archivo in pdfs_sharepoint:
    inventario.append({
        "Archivo": archivo["name"],
        "ID": archivo["id"],
        "eTag": archivo.get("eTag")
    })

df_inventario = pd.DataFrame(inventario)

display(df_inventario)

,Archivo,ID,eTag
0,02_orden_mxn.pdf,01JLPVCZ6VF5TLANRAIRE2XUQVGERKDNYN,"""{B0662FD5-2036-4944-ABD2-153122A1B70D},1"""
1,03_orden_mxn.pdf,01JLPVCZ5IZY3WXULNWJC3TJDZJAPJE33T,"""{6B37CEA8-6DD1-45B2-B9A4-79481E926F73},1"""


In [46]:
import json
from pathlib import Path
from google.colab import files

ruta_registro = Path("resultados/registro_procesados.json")
ruta_registro.parent.mkdir(parents=True, exist_ok=True)

if ruta_registro.exists():
    registro_procesados = json.loads(
        ruta_registro.read_text(encoding="utf-8")
    )

    if registro_procesados["drive_id"] != drive_id:
        raise ValueError("El registro corresponde a otra biblioteca.")

    print("Registro existente cargado.")

else:
    registro_procesados = {
        "drive_id": drive_id,
        "archivos": {}
    }

    for archivo in pdfs_sharepoint:
        registro_procesados["archivos"][archivo["id"]] = {
            "nombre": archivo["name"],
            "eTag": archivo["eTag"]
        }

    contenido = json.dumps(
        registro_procesados,
        ensure_ascii=False,
        indent=2
    )

    ruta_registro.write_text(contenido, encoding="utf-8")
    print("Registro inicial creado.")

print("Archivos registrados:", len(registro_procesados["archivos"]))
print("Ruta:", ruta_registro.resolve())

files.download(str(ruta_registro))

Registro existente cargado.
Archivos registrados: 3
Ruta: /content/resultados/registro_procesados.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
# Leer el registro guardado
registro_procesados = json.loads(
    ruta_registro.read_text(encoding="utf-8")
)

if registro_procesados["drive_id"] != drive_id:
    raise ValueError("El registro corresponde a otra biblioteca.")

guardados = registro_procesados["archivos"]

ids_por_nombre = {
    datos["nombre"].casefold(): id_guardado
    for id_guardado, datos in guardados.items()
}

# Consultar SharePoint
url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root/children"
archivos_actuales = []

while url:
    respuesta = requests.get(url, headers=headers, timeout=30)
    respuesta.raise_for_status()

    pagina = respuesta.json()
    archivos_actuales.extend(pagina["value"])
    url = pagina.get("@odata.nextLink")

comparacion = []
pdfs_pendientes = []

for original in archivos_actuales:
    if "file" not in original or not original["name"].lower().endswith(".pdf"):
        continue

    archivo = original.copy()

    # Buscar primero por ID y después por nombre
    if archivo["id"] in guardados:
        id_anterior = archivo["id"]
    else:
        id_anterior = ids_por_nombre.get(archivo["name"].casefold())

    anterior = guardados.get(id_anterior)

    if anterior is None:
        estado = "Nuevo"
    elif (
        id_anterior != archivo["id"]
        or anterior["eTag"] != archivo["eTag"]
    ):
        estado = "Modificado"
    else:
        estado = "Sin cambios"

    comparacion.append({"Archivo": archivo["name"], "Estado": estado})

    if estado in ["Nuevo", "Modificado"]:
        archivo["id_anterior"] = id_anterior
        pdfs_pendientes.append(archivo)

df_comparacion = pd.DataFrame(comparacion)

display(df_comparacion)
print("PDFs pendientes de procesar:", len(pdfs_pendientes))

,Archivo,Estado
0,02_orden_mxn.pdf,Sin cambios
1,03_orden_mxn.pdf,Sin cambios
2,04_orden_mxn.pdf,Modificado


PDFs pendientes de procesar: 1


In [40]:
carpeta_descargas = Path("pdfs_descargados")
carpeta_descargas.mkdir(exist_ok=True)

encabezados_extraidos = []
partidas_extraidas = []
archivos_extraidos = []

df_ordenes_pendientes = pd.DataFrame()
df_partidas_pendientes = pd.DataFrame()

for archivo in pdfs_pendientes:
    print("Procesando pendiente:", archivo["name"])

    url_descarga = (
        f"https://graph.microsoft.com/v1.0/drives/{drive_id}"
        f"/items/{archivo['id']}/content"
    )

    descarga = requests.get(
        url_descarga, headers=headers, timeout=60
    )
    descarga.raise_for_status()

    ruta_local = carpeta_descargas / archivo["name"]
    ruta_local.write_bytes(descarga.content)

    encabezado, partidas = procesar_pdf(ruta_local)

    encabezados_extraidos.append(encabezado)
    partidas_extraidas.append(partidas)
    archivos_extraidos.append(archivo.copy())

if encabezados_extraidos:
    df_ordenes_pendientes = pd.concat(
        encabezados_extraidos, ignore_index=True
    )
    df_partidas_pendientes = pd.concat(
        partidas_extraidas, ignore_index=True
    )

    print("\nÓrdenes extraídas:", len(df_ordenes_pendientes))
    display(df_ordenes_pendientes)

    print("\nPartidas extraídas:", len(df_partidas_pendientes))
    display(df_partidas_pendientes)
else:
    print("No hay PDFs pendientes de procesar.")

Procesando pendiente: 04_orden_mxn.pdf

Órdenes extraídas: 1


,Orden de compra,Proveedor,Fecha,Moneda,Formato numerico,Subtotal,Impuesto,Total
0,OC-2004,Proveedor Ficticio Sur,2026-11-04,MXN,US,1207.95,193.27,1401.22



Partidas extraídas: 4


,Orden de compra,PN,Descripcion,Cantidad,Precio unitario,Importe
0,OC-2004,000341,Brida de prueba,2,175.25,350.50
1,OC-2004,000342,Empaque de prueba,5,24.10,120.50
2,OC-2004,000343,Valvula de prueba,1,489.75,489.75
3,OC-2004,000344,Soporte de prueba,3,82.40,247.20


In [41]:
def validar_lote(ordenes, partidas):
    if ordenes.empty or partidas.empty:
        raise ValueError("El lote debe contener órdenes y partidas.")

    revisiones = []

    for _, orden in ordenes.iterrows():
        partidas_orden = partidas.loc[
            partidas["Orden de compra"] == orden["Orden de compra"]
        ]

        importes_calculados = (
            partidas_orden["Cantidad"]
            * partidas_orden["Precio unitario"]
        ).round(2)

        importes_correctos = (
            not partidas_orden.empty
            and (
                importes_calculados
                == partidas_orden["Importe"].round(2)
            ).all()
        )

        subtotal_calculado = round(partidas_orden["Importe"].sum(), 2)
        total_calculado = round(orden["Subtotal"] + orden["Impuesto"], 2)

        revisiones.append({
            "Orden de compra": orden["Orden de compra"],
            "Partidas": len(partidas_orden),
            "Importes correctos": bool(importes_correctos),
            "Subtotal correcto": subtotal_calculado == round(orden["Subtotal"], 2),
            "Total correcto": total_calculado == round(orden["Total"], 2)
        })

    return pd.DataFrame(revisiones)


df_validacion_pendientes = validar_lote(
    df_ordenes_pendientes,
    df_partidas_pendientes
)

display(df_validacion_pendientes)

,Orden de compra,Partidas,Importes correctos,Subtotal correcto,Total correcto
0,OC-2004,4,True,True,True


In [62]:
controles = [
    "Importes correctos",
    "Subtotal correcto",
    "Total correcto"
]

if (
    df_validacion_pendientes.empty
    or not df_validacion_pendientes[controles].all().all()
):
    raise ValueError("El lote pendiente debe pasar todas las revisiones.")

ordenes_a_integrar = df_ordenes_pendientes["Orden de compra"]

if ordenes_a_integrar.duplicated().any():
    raise ValueError("Hay números de orden repetidos en el lote pendiente.")

# Conservar las otras órdenes y sus partidas
conservar_ordenes = ~df_ordenes["Orden de compra"].isin(ordenes_a_integrar)
conservar_partidas = ~df_partidas_consolidadas["Orden de compra"].isin(
    ordenes_a_integrar
)

# Incorporar las versiones recién procesadas
df_ordenes_actualizadas = pd.concat(
    [df_ordenes.loc[conservar_ordenes], df_ordenes_pendientes],
    ignore_index=True
)

df_partidas_actualizadas = pd.concat(
    [
        df_partidas_consolidadas.loc[conservar_partidas],
        df_partidas_pendientes
    ],
    ignore_index=True
)

# Validar el consolidado completo
df_validacion_actualizada = validar_lote(
    df_ordenes_actualizadas,
    df_partidas_actualizadas
)

print("Órdenes del consolidado:", len(df_ordenes_actualizadas))
print("Partidas del consolidado:", len(df_partidas_actualizadas))

display(df_validacion_actualizada)

Órdenes del consolidado: 3
Partidas del consolidado: 9


,Orden de compra,Partidas,Importes correctos,Subtotal correcto,Total correcto
0,OC-2002,2,True,True,True
1,OC-2003,3,True,True,True
2,OC-2004,4,True,True,True


In [61]:
if (
    df_validacion_actualizada.empty
    or not df_validacion_actualizada[controles].all().all()
):
    raise ValueError("El consolidado debe pasar todas las revisiones.")

registro_actualizado = json.loads(
    ruta_registro.read_text(encoding="utf-8")
)

if registro_actualizado["drive_id"] != drive_id:
    raise ValueError("El registro corresponde a otra biblioteca.")

# Guardar el Excel actualizado mediante un archivo temporal
excel_temporal = ruta_excel_lote.with_name("consolidado_temporal.xlsx")

with pd.ExcelWriter(excel_temporal, engine="openpyxl") as escritor:
    df_ordenes_actualizadas.to_excel(escritor, sheet_name="Ordenes", index=False)
    df_partidas_actualizadas.to_excel(escritor, sheet_name="Partidas", index=False)
    df_validacion_actualizada.to_excel(escritor, sheet_name="Validaciones", index=False)

excel_temporal.replace(ruta_excel_lote)

# Registrar los archivos extraídos y validados
for archivo in archivos_extraidos:
    registro_actualizado["archivos"][archivo["id"]] = {
        "nombre": archivo["name"],
        "eTag": archivo["eTag"]
    }

registro_temporal = ruta_registro.with_suffix(".tmp")
registro_temporal.write_text(
    json.dumps(registro_actualizado, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
registro_temporal.replace(ruta_registro)

# Actualizar las tablas de trabajo
df_ordenes = df_ordenes_actualizadas.copy()
df_partidas_consolidadas = df_partidas_actualizadas.copy()
df_validacion_lote = df_validacion_actualizada.copy()
registro_procesados = registro_actualizado

print("Órdenes guardadas:", len(df_ordenes))
print("Partidas guardadas:", len(df_partidas_consolidadas))
print("Archivos registrados:", len(registro_procesados["archivos"]))

files.download(str(ruta_excel_lote))
files.download(str(ruta_registro))

Órdenes guardadas: 3
Partidas guardadas: 9
Archivos registrados: 3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
registro_revision = json.loads(
    ruta_registro.read_text(encoding="utf-8")
)

respuesta_revision = requests.get(
    f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root/children",
    headers=headers,
    timeout=30
)
respuesta_revision.raise_for_status()

nombres_guardados = {
    datos["nombre"].casefold()
    for datos in registro_revision["archivos"].values()
}

revision = []

for archivo in respuesta_revision.json()["value"]:
    if "file" not in archivo:
        continue

    if not archivo["name"].casefold().startswith("04_orden_mxn"):
        continue

    anterior = registro_revision["archivos"].get(archivo["id"])

    if anterior is None:
        estado = "Nuevo"
    elif anterior["eTag"] != archivo["eTag"]:
        estado = "Modificado"
    else:
        estado = "Sin cambios"

    revision.append({
        "Archivo real": archivo["name"],
        "ID reconocido": anterior is not None,
        "Nombre reconocido": archivo["name"].casefold() in nombres_guardados,
        "Estado actual": estado
    })

print("Registro utilizado:", ruta_registro.resolve())
print("Archivos registrados:", len(registro_revision["archivos"]))

display(pd.DataFrame(revision))

Registro utilizado: /content/resultados/registro_procesados.json
Archivos registrados: 3


,Archivo real,ID reconocido,Nombre reconocido,Estado actual
0,04_orden_mxn.pdf,False,True,Nuevo
